In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE
# This will prompt you to authorize access to your Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# --- CONFIGURATION ---
BASE_URL = "https://www.federalreserve.gov"
START_URL = "https://www.federalreserve.gov/monetarypolicy/fomc_historical_year.htm"
TARGET_YEARS = ["2019", "2018", "2017", "2016", "2015", "2014", "2013", "2012", "2011"]

# 2. DEFINE THE DIRECTORY PATH
# In Colab, your 'My Drive' is located at /content/drive/MyDrive/
# Note: I have combined "HEC Thesis" with your target folder "fomc_transcripts_final"
DOWNLOAD_DIR = "/content/drive/MyDrive/HEC Thesis/fomc_transcripts_final"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Ensure the directory exists in your Google Drive
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)
    print(f"Created directory: {DOWNLOAD_DIR}")

def run_deep_extraction():
    session = requests.Session()

    # 1. Get the main list of years
    print(f"--- Accessing Main History Page ---")
    try:
        response = session.get(START_URL, headers=HEADERS)
        soup = BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"Critical Error: {e}")
        return

    # Filter for the years we want
    year_pages = {}
    for link in soup.find_all('a', href=True):
        text = link.get_text().strip()
        if text in TARGET_YEARS:
            year_pages[text] = urljoin(BASE_URL, link['href'])

    # 2. Process each Year
    for year, year_url in year_pages.items():
        print(f"\n=== Processing Year: {year} ===")

        try:
            year_res = session.get(year_url, headers=HEADERS)
            year_soup = BeautifulSoup(year_res.text, 'html.parser')

            # Find the intermediate "Press Conference" HTML pages
            press_conf_pages = []
            for link in year_soup.find_all('a', href=True):
                if "Press Conference" in link.get_text() or "fomcpresconf" in link['href']:
                    full_link = urljoin(BASE_URL, link['href'])
                    if not full_link.endswith('.pdf'):
                        if full_link not in press_conf_pages:
                            press_conf_pages.append(full_link)

            print(f"  Found {len(press_conf_pages)} press conference pages. Digging deeper...")

            # STEP 2: Go into each Press Conference Page to find the PDF
            for pc_url in press_conf_pages:
                try:
                    pc_res = session.get(pc_url, headers=HEADERS)
                    pc_soup = BeautifulSoup(pc_res.text, 'html.parser')

                    pdf_found = False
                    for pdf_link in pc_soup.find_all('a', href=True):
                        href = pdf_link['href']
                        text = pdf_link.get_text().lower()

                        # Logic: matches 'transcript' AND ends in '.pdf'
                        if 'transcript' in text and href.lower().endswith('.pdf'):
                            final_pdf_url = urljoin(BASE_URL, href)
                            filename = final_pdf_url.split('/')[-1]
                            save_path = os.path.join(DOWNLOAD_DIR, filename)

                            # Check if file already exists to avoid redundant downloads
                            if os.path.exists(save_path):
                                print(f"    -> Already exists: {filename}")
                                pdf_found = True
                                break

                            print(f"    -> Downloading: {filename}")

                            with open(save_path, 'wb') as f:
                                f.write(session.get(final_pdf_url, headers=HEADERS).content)

                            pdf_found = True
                            time.sleep(1) # Be polite
                            break

                    if not pdf_found:
                        print(f"    [!] No PDF transcript link found on {pc_url}")

                except Exception as e:
                    print(f"    Error accessing sub-page: {e}")

        except Exception as e:
            print(f"  Error accessing year page {year}: {e}")

if __name__ == "__main__":
    run_deep_extraction()
    print(f"\n--- Done! Check your Google Drive folder: HEC Thesis/fomc_transcripts_final ---")

Mounted at /content/drive
Created directory: /content/drive/MyDrive/HEC Thesis/fomc_transcripts_final
--- Accessing Main History Page ---

=== Processing Year: 2019 ===
  Found 8 press conference pages. Digging deeper...
    -> Downloading: FOMCpresconf20190130.pdf
    -> Downloading: FOMCpresconf20190320.pdf
    -> Downloading: FOMCpresconf20190501.pdf
    -> Downloading: FOMCpresconf20190619.pdf
    -> Downloading: FOMCpresconf20190731.pdf
    -> Downloading: FOMCpresconf20190918.pdf
    -> Downloading: FOMCpresconf20191030.pdf
    -> Downloading: FOMCpresconf20191211.pdf

=== Processing Year: 2018 ===
  Found 4 press conference pages. Digging deeper...
    -> Downloading: FOMCpresconf20180321.pdf
    -> Downloading: FOMCpresconf20180613.pdf
    -> Downloading: FOMCpresconf20180926.pdf
    -> Downloading: FOMCpresconf20181219.pdf

=== Processing Year: 2017 ===
  Found 4 press conference pages. Digging deeper...
    -> Downloading: FOMCpresconf20170315.pdf
    -> Downloading: FOMCpres

KeyboardInterrupt: 

In [ ]:
!pip install pdfplumber

import os
import json
import pdfplumber
import re

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/HEC Thesis/fomc_transcripts_final"
# New destination folder for the individual JSONs
OUTPUT_DIR = "/content/drive/MyDrive/HEC Thesis/structured_json"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def clean_fomc_text(text):
    """Removes common PDF artifacts and recurring headers."""
    # Remove page numbers and headers
    text = re.sub(r'\n\s*\d+\s+of\s+\d+\s*\n', '\n', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)

    header_patterns = [
        r"Transcript of Chair .* Press Conference",
        r"FEDERAL RESERVE BOARD",
        r"RESERVE BOARD OF GOVERNORS"
    ]
    for pattern in header_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

def split_remarks_and_qa(full_text):
    """Separates scripted remarks from the Q&A session."""
    qa_markers = [
        "QUESTIONS AND ANSWERS",
        "thank you. we'll now take questions",
        "i'm happy to take your questions"
    ]

    remarks = full_text
    qa = ""

    for marker in qa_markers:
        if marker.upper() in full_text.upper():
            parts = re.split(marker, full_text, flags=re.IGNORECASE)
            remarks = parts[0]
            qa = marker + parts[1] if len(parts) > 1 else ""
            break

    return remarks.strip(), qa.strip()

def process_to_individual_files():
    pdf_files = [f for f in os.listdir(BASE_PATH) if f.endswith('.pdf')]
    print(f"Starting processing of {len(pdf_files)} files...")

    for filename in pdf_files:
        filepath = os.path.join(BASE_PATH, filename)

        # Create a clean name for the JSON (e.g., 'fomcpresconf20190320.json')
        json_filename = filename.replace('.pdf', '.json').replace('.PDF', '.json')
        output_path = os.path.join(OUTPUT_DIR, json_filename)

        print(f"Processing: {filename} -> {json_filename}")

        try:
            full_raw_text = ""
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages:
                    # Ignore top/bottom 50 units for headers/footers
                    page_crop = page.within_bbox((0, 50, page.width, page.height - 50))
                    page_text = page_crop.extract_text()
                    if page_text:
                        full_raw_text += page_text + "\n"

            cleaned_text = clean_fomc_text(full_raw_text)
            remarks, qa = split_remarks_and_qa(cleaned_text)

            date_match = re.search(r'\d{8}', filename)
            date_str = date_match.group(0) if date_match else "Unknown"

            entry = {
                "source": filename,
                "date": date_str,
                "prepared_remarks": remarks,
                "qa_session": qa,
                "word_counts": {
                    "total": len(cleaned_text.split()),
                    "remarks": len(remarks.split()),
                    "qa": len(qa.split())
                }
            }

            # Save the individual JSON file
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(entry, f, indent=4, ensure_ascii=False)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print(f"\nSUCCESS! Individual JSONs are in: {OUTPUT_DIR}")

# RUN IT
process_to_individual_files()

Starting processing of 40 files...
Processing: FOMCpresconf20190130.pdf -> FOMCpresconf20190130.json
Processing: FOMCpresconf20190320.pdf -> FOMCpresconf20190320.json
Processing: FOMCpresconf20190501.pdf -> FOMCpresconf20190501.json
Processing: FOMCpresconf20190619.pdf -> FOMCpresconf20190619.json
Processing: FOMCpresconf20190731.pdf -> FOMCpresconf20190731.json
Processing: FOMCpresconf20190918.pdf -> FOMCpresconf20190918.json
Processing: FOMCpresconf20191030.pdf -> FOMCpresconf20191030.json
Processing: FOMCpresconf20191211.pdf -> FOMCpresconf20191211.json
Processing: FOMCpresconf20180321.pdf -> FOMCpresconf20180321.json
Processing: FOMCpresconf20180613.pdf -> FOMCpresconf20180613.json
Processing: FOMCpresconf20180926.pdf -> FOMCpresconf20180926.json
Processing: FOMCpresconf20181219.pdf -> FOMCpresconf20181219.json
Processing: FOMCpresconf20170315.pdf -> FOMCpresconf20170315.json
Processing: FOMCpresconf20170614.pdf -> FOMCpresconf20170614.json
Processing: FOMCpresconf20170920.pdf -> F

In [ ]:
# ... (Keep your imports and configuration at the top unchanged) ...

def clean_filename(fname):
    """Removes illegal characters from filenames."""
    # Remove query strings (e.g., 'file.pdf?v=1' -> 'file.pdf')
    fname = fname.split('?')[0]
    # Remove other illegal characters
    fname = re.sub(r'[<>:"/\\|?*]', '', fname)
    return fname

def run_speech_scraper():
    session = requests.Session()

    for year in YEARS:
        print(f"\n=== Processing Year: {year} ===")

        archive_url = f"{BASE_URL}/newsevents/speech/{year}-speeches.htm"
        main_url = f"{BASE_URL}/newsevents/speeches.htm"

        target_url = archive_url
        is_current_year_fallback = False

        try:
            res = session.get(archive_url, headers=HEADERS)
            if res.status_code != 200:
                if year >= 2024:
                    print(f"  Archive not found. Trying main page...")
                    target_url = main_url
                    res = session.get(target_url, headers=HEADERS)
                    is_current_year_fallback = True
                else:
                    print(f"  [!] Could not access archive for {year} (Status: {res.status_code})")
                    continue
        except Exception as e:
            print(f"  Connection error: {e}")
            continue

        soup = BeautifulSoup(res.text, 'html.parser')

        if is_current_year_fallback:
             articles = soup.find_all('div', class_=['row', 'eventlist__item'])
        else:
             articles = soup.find_all('div', class_='row')

        count_for_year = 0

        for article in articles:
            # 1. Extract Link
            link_tag = article.find('a', href=True)
            if not link_tag: continue

            href = link_tag['href']
            if "/speech/" not in href: continue

            full_speech_url = urljoin(BASE_URL, href)
            article_text = article.get_text(" ", strip=True)

            if is_current_year_fallback and str(year) not in article_text:
                continue

            # 2. Determine Folder
            target_folder = get_category_folder(article_text)

            try:
                # 3. Enter Speech Page
                speech_res = session.get(full_speech_url, headers=HEADERS)
                speech_soup = BeautifulSoup(speech_res.text, 'html.parser')

                # 4. Find Best PDF Link
                pdf_href = find_best_pdf_link(speech_soup)

                if pdf_href:
                    pdf_url = urljoin(BASE_URL, pdf_href)

                    # --- ROBUST FILENAME CLEANING ---
                    raw_filename = pdf_url.split('/')[-1]
                    clean_name = clean_filename(raw_filename)
                    save_name = f"{year}_{clean_name}"
                    save_path = os.path.join(target_folder, save_name)

                    # --- ROBUST SAVING (Fixes Errno 2) ---
                    if not os.path.exists(save_path):
                        print(f"    -> Downloading: {save_name}")

                        # FORCE folder check immediately before writing
                        if not os.path.exists(target_folder):
                            os.makedirs(target_folder, exist_ok=True)

                        try:
                            pdf_content = session.get(pdf_url, headers=HEADERS).content
                            with open(save_path, 'wb') as f:
                                f.write(pdf_content)
                            count_for_year += 1
                        except Exception as e:
                            # If it fails, wait 2 seconds and retry (helps with Drive latency)
                            print(f"       [!] Save failed. Retrying in 2s... ({e})")
                            time.sleep(2)
                            if not os.path.exists(target_folder):
                                os.makedirs(target_folder, exist_ok=True)
                            with open(save_path, 'wb') as f:
                                f.write(pdf_content)
                            print("       [+] Retry successful.")

                        time.sleep(0.5)

            except Exception as e:
                print(f"    Error processing {full_speech_url}: {e}")

        print(f"  Finished {year}: Downloaded {count_for_year} PDFs.")

# Run the updated function
if __name__ == "__main__":
    run_speech_scraper()
    print("\n--- Done! Check 'HEC Thesis/speeches' folder ---")


=== Processing Year: 2011 ===
    -> Downloading: 2011_yellen20111129a1.pdf
    -> Downloading: 2011_yellen20111111a.pdf
    -> Downloading: 2011_bernanke20111110a.pdf
    -> Downloading: 2011_tarullo20111109a.pdf
    -> Downloading: 2011_bernanke20111109a.pdf
    -> Downloading: 2011_tarullo20111104a.pdf
    -> Downloading: 2011_duke20111022a.pdf
    -> Downloading: 2011_yellen20111021a.pdf
    -> Downloading: 2011_tarullo20111020a.pdf
    -> Downloading: 2011_bernanke20111018a.pdf
    -> Downloading: 2011_raskin20111004a.pdf
    -> Downloading: 2011_bernanke20110928a.pdf
    -> Downloading: 2011_raskin20110926a.pdf
    -> Downloading: 2011_bernanke20110915a1.pdf
    -> Downloading: 2011_tarullo20110915a1.pdf
    -> Downloading: 2011_bernanke20110908a1.pdf
    -> Downloading: 2011_duke20110901a.pdf
    -> Downloading: 2011_bernanke20110826a.pdf
    -> Downloading: 2011_raskin20110629a.pdf
    -> Downloading: 2011_bernanke20110614a.pdf
    -> Downloading: 2011_yellen20110609a.pdf
    

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re
from google.colab import drive

# 1. MOUNT DRIVE
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURATION ---
BASE_ROOT = "/content/drive/MyDrive/HEC Thesis/fomc_transcripts_final"
BASE_URL = "https://www.federalreserve.gov"
CALENDAR_URL = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"
TARGET_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

if not os.path.exists(BASE_ROOT):
    os.makedirs(BASE_ROOT, exist_ok=True)

def clean_filename(fname):
    fname = fname.split('?')[0]
    fname = re.sub(r'[<>:"/\\|?*]', '', fname)
    return fname

def run_recent_scraper():
    session = requests.Session()
    print(f"--- Accessing Main Calendar: {CALENDAR_URL} ---")

    try:
        res = session.get(CALENDAR_URL, headers=HEADERS)
        soup = BeautifulSoup(res.text, 'html.parser')

        # 1. Find all "Press Conference" links (HTML pages, not PDFs yet)
        # They usually contain 'fomcpresconf' in the href
        press_conf_pages = []

        for a in soup.find_all('a', href=True):
            href = a['href']
            text = a.get_text().strip().lower()

            # We want the HTML page for the press conference
            if 'presconf' in href.lower() and not href.lower().endswith('.pdf'):
                # Extract year from URL (e.g., fomcpresconf20210317.htm)
                # We look for a 4-digit year in the href
                year_match = re.search(r'20\d{2}', href)
                if year_match:
                    year = int(year_match.group(0))
                    if year in TARGET_YEARS:
                        full_url = urljoin(BASE_URL, href)
                        if full_url not in press_conf_pages:
                            press_conf_pages.append(full_url)

        print(f"Found {len(press_conf_pages)} press conference pages for {TARGET_YEARS}. Processing...")

        # 2. Visit each Press Conf Page to get the PDF
        for pc_url in press_conf_pages:
            try:
                # Extract date/name for logging
                page_name = pc_url.split('/')[-1]
                print(f"\nChecking: {page_name}")

                pc_res = session.get(pc_url, headers=HEADERS)
                pc_soup = BeautifulSoup(pc_res.text, 'html.parser')

                # Look for the PDF link inside this page
                # Usually says "Transcript (PDF)"
                pdf_found = False
                for a in pc_soup.find_all('a', href=True):
                    if 'transcript' in a.get_text().lower() and a['href'].lower().endswith('.pdf'):
                        pdf_url = urljoin(BASE_URL, a['href'])

                        # Filename consistency
                        # Server name: FOMCpresconf20210317.pdf
                        raw_filename = pdf_url.split('/')[-1]
                        clean_name = clean_filename(raw_filename)
                        save_path = os.path.join(BASE_ROOT, clean_name)

                        if not os.path.exists(save_path):
                            print(f"  -> Downloading: {clean_name}")
                            pdf_content = session.get(pdf_url, headers=HEADERS).content
                            with open(save_path, 'wb') as f:
                                f.write(pdf_content)
                            time.sleep(1)
                        else:
                            print(f"  -> Skipping {clean_name} (Exists)")

                        pdf_found = True
                        break # Only need one transcript per page

                if not pdf_found:
                    print(f"  [!] No PDF transcript found on {page_name}")

                time.sleep(0.5)

            except Exception as e:
                print(f"  Error processing {pc_url}: {e}")

    except Exception as e:
        print(f"Critical Error: {e}")

if __name__ == "__main__":
    run_recent_scraper()
    print("\n--- Done! Recent transcripts added. ---")

--- Accessing Main Calendar: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm ---
Found 40 press conference pages for [2021, 2022, 2023, 2024, 2025, 2026]. Processing...

Checking: fomcpresconf20250129.htm
  -> Downloading: FOMCpresconf20250129.pdf

Checking: fomcpresconf20250319.htm
  -> Downloading: FOMCpresconf20250319.pdf

Checking: fomcpresconf20250507.htm
  -> Downloading: FOMCpresconf20250507.pdf

Checking: fomcpresconf20250618.htm
  -> Downloading: FOMCpresconf20250618.pdf

Checking: fomcpresconf20250730.htm
  -> Downloading: FOMCpresconf20250730.pdf

Checking: fomcpresconf20250917.htm
  -> Downloading: FOMCpresconf20250917.pdf

Checking: fomcpresconf20251029.htm
  -> Downloading: FOMCpresconf20251029.pdf

Checking: fomcpresconf20251210.htm
  -> Downloading: FOMCpresconf20251210.pdf

Checking: fomcpresconf20240131.htm
  -> Downloading: FOMCpresconf20240131.pdf

Checking: fomcpresconf20240320.htm
  -> Downloading: FOMCpresconf20240320.pdf

Checking: fomcpresconf20

In [ ]:
import os
import json
import pdfplumber
import re

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/HEC Thesis/fomc_transcripts_final"
OUTPUT_DIR = "/content/drive/MyDrive/HEC Thesis/structured_json"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def clean_fomc_text(text):
    """Removes common PDF artifacts and recurring headers."""
    text = re.sub(r'\n\s*\d+\s+of\s+\d+\s*\n', '\n', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)

    header_patterns = [
        r"Transcript of Chair .* Press Conference",
        r"FEDERAL RESERVE BOARD",
        r"RESERVE BOARD OF GOVERNORS",
        r"FINAL TRANSCRIPT"
    ]
    for pattern in header_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

def split_remarks_and_qa(full_text):
    """Separates scripted remarks from the Q&A session."""
    # Updated markers to catch more variations
    qa_markers = [
        "QUESTIONS AND ANSWERS",
        "thank you. we'll now take questions",
        "i'm happy to take your questions",
        "we will now take questions",
        "let's go to questions"
    ]

    remarks = full_text
    qa = ""

    for marker in qa_markers:
        if marker.upper() in full_text.upper():
            parts = re.split(marker, full_text, flags=re.IGNORECASE, maxsplit=1)
            remarks = parts[0]
            qa = marker + parts[1] if len(parts) > 1 else ""
            break

    return remarks.strip(), qa.strip()

def process_press_conferences():
    pdf_files = [f for f in os.listdir(BASE_PATH) if f.lower().endswith('.pdf')]
    print(f"Found {len(pdf_files)} PDFs in source folder.")

    processed_count = 0
    skipped_count = 0

    for filename in pdf_files:
        filepath = os.path.join(BASE_PATH, filename)

        # Create JSON filename
        json_filename = filename.replace('.pdf', '.json').replace('.PDF', '.json')
        output_path = os.path.join(OUTPUT_DIR, json_filename)

        # SKIP IF EXISTS (Efficiency check)
        if os.path.exists(output_path):
            skipped_count += 1
            continue

        print(f"Processing: {filename}")

        try:
            full_raw_text = ""
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages:
                    # Crop headers/footers
                    page_crop = page.within_bbox((0, 50, page.width, page.height - 50))
                    page_text = page_crop.extract_text()
                    if page_text:
                        full_raw_text += page_text + "\n"

            cleaned_text = clean_fomc_text(full_raw_text)
            remarks, qa = split_remarks_and_qa(cleaned_text)

            # Extract Date from filename (e.g., FOMCpresconf20190320.pdf)
            date_match = re.search(r'20\d{6}', filename) # Looks for 20190320
            date_str = date_match.group(0) if date_match else "Unknown"

            entry = {
                "source_filename": filename,
                "date": date_str,
                "type": "Press Conference",
                "prepared_remarks": remarks,
                "qa_session": qa,
                "word_counts": {
                    "total": len(cleaned_text.split()),
                    "remarks": len(remarks.split()),
                    "qa": len(qa.split())
                }
            }

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(entry, f, indent=4, ensure_ascii=False)

            processed_count += 1

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print(f"\n--- DONE ---")
    print(f"New files processed: {processed_count}")
    print(f"Old files skipped: {skipped_count}")
    print(f"Check folder: {OUTPUT_DIR}")

# RUN IT
process_press_conferences()

Found 80 PDFs in source folder.
Processing: FOMCpresconf20250129.pdf
Processing: FOMCpresconf20250319.pdf
Processing: FOMCpresconf20250507.pdf
Processing: FOMCpresconf20250618.pdf
Processing: FOMCpresconf20250730.pdf
Processing: FOMCpresconf20250917.pdf
Processing: FOMCpresconf20251029.pdf
Processing: FOMCpresconf20251210.pdf
Processing: FOMCpresconf20240131.pdf
Processing: FOMCpresconf20240320.pdf
Processing: FOMCpresconf20240501.pdf
Processing: FOMCpresconf20240612.pdf
Processing: fomcpresconf20240731.pdf
Processing: FOMCpresconf20240918.pdf
Processing: FOMCpresconf20241107.pdf
Processing: FOMCpresconf20241218.pdf
Processing: FOMCpresconf20230201.pdf
Processing: FOMCpresconf20230322.pdf
Processing: FOMCpresconf20230503.pdf
Processing: FOMCpresconf20230614.pdf
Processing: FOMCpresconf20230726.pdf
Processing: FOMCpresconf20230920.pdf
Processing: FOMCpresconf20231101.pdf
Processing: FOMCpresconf20231213.pdf
Processing: FOMCpresconf20220126.pdf
Processing: FOMCpresconf20220316.pdf
Proces

In [ ]:
import os
import json
import pdfplumber
import re

# --- CONFIGURATION ---
SPEECHES_ROOT = "/content/drive/MyDrive/HEC Thesis/speeches"
OUTPUT_ROOT = "/content/drive/MyDrive/HEC Thesis/structured_json_speeches"

def clean_speech_text(text):
    """Specific cleaning for speeches."""
    # Remove page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    # Fix broken whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def process_speeches():
    print(f"Scanning speech folders in: {SPEECHES_ROOT}")

    # Walk through all subfolders (chair, vice_chair, others)
    for root, dirs, files in os.walk(SPEECHES_ROOT):
        for filename in files:
            if not filename.lower().endswith('.pdf'):
                continue

            # Identify the category (e.g., 'chair') from the folder name
            category = os.path.basename(root)

            # Create the corresponding OUTPUT folder
            target_folder = os.path.join(OUTPUT_ROOT, category)
            if not os.path.exists(target_folder):
                os.makedirs(target_folder)

            # Paths
            source_path = os.path.join(root, filename)
            json_filename = filename.replace('.pdf', '.json').replace('.PDF', '.json')
            output_path = os.path.join(target_folder, json_filename)

            # Skip if already done
            if os.path.exists(output_path):
                continue

            print(f"Processing ({category}): {filename}")

            try:
                full_raw_text = ""
                with pdfplumber.open(source_path) as pdf:
                    for page in pdf.pages:
                        # Slightly less aggressive crop for speeches as margins vary
                        page_crop = page.within_bbox((0, 40, page.width, page.height - 40))
                        page_text = page_crop.extract_text()
                        if page_text:
                            full_raw_text += page_text + "\n"

                cleaned_text = clean_speech_text(full_raw_text)

                # Extract Metadata from our saved filename format: "YYYY_originalName.pdf"
                # Example: "2011_bernanke_speech.pdf"
                year_part = filename.split('_')[0]
                year = year_part if year_part.isdigit() else "Unknown"

                entry = {
                    "source_filename": filename,
                    "year": year,
                    "category": category, # chair, vice_chair, or others
                    "type": "Speech",
                    "text": cleaned_text,
                    "word_count": len(cleaned_text.split())
                }

                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(entry, f, indent=4, ensure_ascii=False)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    print(f"\n--- DONE ---")
    print(f"Speeches processed and saved to: {OUTPUT_ROOT}")

# RUN IT
process_speeches()

Scanning speech folders in: /content/drive/MyDrive/HEC Thesis/speeches
Processing (vice_chair): 2011_yellen20111129a1.pdf
Processing (vice_chair): 2011_yellen20111111a.pdf
Processing (vice_chair): 2011_yellen20111021a.pdf
Processing (vice_chair): 2011_yellen20110609a.pdf
Processing (vice_chair): 2011_yellen20110601a.pdf
Processing (vice_chair): 2011_yellen20110506a.pdf
Processing (vice_chair): 2011_yellen20110411a.pdf
Processing (vice_chair): 2011_yellen20110304a.pdf
Processing (vice_chair): 2011_yellen20110225a.pdf
Processing (vice_chair): 2011_yellen20110108a.pdf
Processing (vice_chair): 2012_stein20121217a.pdf
Processing (vice_chair): 2012_yellen20121113a.pdf
Processing (vice_chair): 2012_yellen20121011a.pdf
Processing (vice_chair): 2012_yellen20120606a.pdf
Processing (vice_chair): 2012_yellen20120411a.pdf
Processing (vice_chair): 2013_bernanke20131216b.pdf
Processing (vice_chair): 2013_yellen20130602a.pdf
Processing (vice_chair): 2013_yellen20130416a.pdf
Processing (vice_chair): 20

In [ ]:
import pandas as pd
import pandas_datareader.data as web
import datetime
import os

# --- CONFIGURATION ---
# Buffer start to calculate YoY change correctly
DOWNLOAD_START = "2010-01-01"
TARGET_START = "2011-01-01"
END_DATE = datetime.datetime.now()

OUTPUT_PATH = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data.csv"

# Added 'CPIAUCSL' for CPI
SERIES_IDS = ['FEDFUNDS', 'PCEPI', 'CPIAUCSL', 'GDPC1', 'GDPPOT']

def get_taylor_rule_data():
    print(f"--- Fetching Macro Data (Buffer start: {DOWNLOAD_START}) ---")

    try:
        df = web.DataReader(SERIES_IDS, 'fred', DOWNLOAD_START, END_DATE)
    except Exception as e:
        print(f"Error connecting to FRED: {e}")
        return None

    # 1. CALCULATE INFLATION (YoY % Change)
    # PCE Inflation
    df['inflation_pce'] = df['PCEPI'].pct_change(periods=12, fill_method=None) * 100

    # CPI Inflation (New)
    df['inflation_cpi'] = df['CPIAUCSL'].pct_change(periods=12, fill_method=None) * 100

    # 2. OUTPUT GAP
    quarterly_cols = ['GDPC1', 'GDPPOT']
    df_q = df[quarterly_cols].dropna().resample('QS').mean()
    df_q['output_gap'] = 100 * (df_q['GDPC1'] - df_q['GDPPOT']) / df_q['GDPPOT']

    # Upsample to Monthly
    df_q_monthly = df_q[['output_gap']].resample('MS').first()
    df_q_interpolated = df_q_monthly.interpolate(method='linear')

    # Join back
    df = df.join(df_q_interpolated, rsuffix='_calc')

    # 3. CLEAN AND SAVE
    # Select all relevant columns including the new CPI
    final_cols = ['FEDFUNDS', 'inflation_pce', 'inflation_cpi', 'output_gap']
    df_clean = df[final_cols].dropna()

    # Filter for your thesis timeline
    final_df = df_clean[df_clean.index >= TARGET_START]

    print("\n--- FIRST 5 ROWS ---")
    print(final_df.head())

    return final_df

if __name__ == "__main__":
    macro_df = get_taylor_rule_data()

    if macro_df is not None:
        folder = os.path.dirname(OUTPUT_PATH)
        if not os.path.exists(folder):
            os.makedirs(folder)

        macro_df.to_csv(OUTPUT_PATH)
        print(f"\nSUCCESS! File saved to: {OUTPUT_PATH}")

--- Fetching Macro Data (Buffer start: 2010-01-01) ---

--- FIRST 5 ROWS ---
            FEDFUNDS  inflation_pce  inflation_cpi  output_gap
DATE                                                          
2011-01-01      0.17       1.559865       1.700783   -3.643767
2011-02-01      0.16       1.845031       2.124898   -3.554774
2011-03-01      0.14       2.110546       2.619242   -3.465781
2011-04-01      0.10       2.488097       3.077234   -3.376788
2011-05-01      0.09       2.766247       3.458972   -3.515149

SUCCESS! File saved to: /content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data.csv


In [ ]:
import pandas as pd
import requests
import datetime
import os

# --- CONFIGURATION ---
FRED_API_KEY = "94afca2e4faf7c6dc2cbf15deca6ce7b"
OUTPUT_PATH = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data_vintage.csv"

# FOMC meeting dates — you need to fetch data as of each of these dates
# Full list available at: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
FOMC_DATES = [
    # 2011
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22",
    "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    # 2012
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20",
    "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    # 2013
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19",
    "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    # 2014
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18",
    "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    # 2015
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17",
    "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    # 2016
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15",
    "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    # 2017
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14",
    "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    # 2018
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13",
    "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    # 2019
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19",
    "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29",
    "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05",
    "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16",
    "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15",
    "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14",
    "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12",
    "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18",
    "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
]

SERIES_IDS = {
    'FEDFUNDS':  'fedfunds',
    'PCEPI':     'inflation_pce_raw',
    'CPIAUCSL':  'inflation_cpi_raw',
    'GDPC1':     'gdp_real',
    'GDPPOT':    'gdp_pot'
}

def get_vintage_observation(series_id, as_of_date, api_key):
    """
    Fetches the most recently available observation of a series
    as it would have looked on a specific date — no future revisions.

    realtime_end = as_of_date means: give me data as it existed on this date
    observation_end = as_of_date means: only observations up to this date
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id':      series_id,
        'api_key':        api_key,
        'file_type':      'json',
        'realtime_start': '2000-01-01',   # vintage start
        'realtime_end':   as_of_date,     # vintage as of meeting date — KEY PARAMETER
        'observation_start': '2010-01-01',
        'observation_end':   as_of_date,
        'sort_order':     'desc',
        'limit':          24             # get last 12 obs for YoY calculation
    }

    response = requests.get(url, params=params)
    data = response.json()

    if 'observations' not in data or len(data['observations']) == 0:
        return None

    observations = pd.DataFrame(data['observations'])
    observations['value'] = pd.to_numeric(observations['value'], errors='coerce')
    observations['date'] = pd.to_datetime(observations['date'])
    observations = observations.set_index('date').sort_index()

    return observations['value']


def get_realtime_taylor_data(fomc_dates, api_key):
    """
    For each FOMC meeting date, fetches the vintage of each macro series
    that would have been available to policymakers at that time.
    """
    records = []

    for meeting_date in fomc_dates:
        print(f"Fetching vintage data as of {meeting_date}...")
        record = {'date': meeting_date}

        # Fed Funds Rate
        ffr = get_vintage_observation('FEDFUNDS', meeting_date, api_key)
        if ffr is not None and len(ffr) >= 1:
            record['FEDFUNDS'] = ffr.iloc[-1]

        # PCE Inflation (YoY)
        pce = get_vintage_observation('PCEPI', meeting_date, api_key)
        if pce is not None and len(pce) >= 13:
            record['inflation_pce'] = (pce.iloc[-1] / pce.iloc[-13] - 1) * 100

        # CPI Inflation (YoY)
        cpi = get_vintage_observation('CPIAUCSL', meeting_date, api_key)
        if cpi is not None and len(cpi) >= 13:
            record['inflation_cpi'] = (cpi.iloc[-1] / cpi.iloc[-13] - 1) * 100

        # Output Gap (GDP real vs potential)
        gdp = get_vintage_observation('GDPC1', meeting_date, api_key)
        pot = get_vintage_observation('GDPPOT', meeting_date, api_key)
        if gdp is not None and pot is not None and len(gdp) >= 1 and len(pot) >= 1:
            record['output_gap'] = 100 * (gdp.iloc[-1] - pot.iloc[-1]) / pot.iloc[-1]

        records.append(record)

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    return df


if __name__ == "__main__":
    macro_df = get_realtime_taylor_data(FOMC_DATES, FRED_API_KEY)
    print(macro_df.head())
    macro_df.to_csv(OUTPUT_PATH)
    print(f"Saved to {OUTPUT_PATH}")

Fetching vintage data as of 2011-01-26...
Fetching vintage data as of 2011-03-15...
Fetching vintage data as of 2011-04-27...
Fetching vintage data as of 2011-06-22...
Fetching vintage data as of 2011-08-09...
Fetching vintage data as of 2011-09-21...
Fetching vintage data as of 2011-11-02...
Fetching vintage data as of 2011-12-13...
Fetching vintage data as of 2012-01-25...
Fetching vintage data as of 2012-03-13...
Fetching vintage data as of 2012-04-25...
Fetching vintage data as of 2012-06-20...
Fetching vintage data as of 2012-08-01...
Fetching vintage data as of 2012-09-13...
Fetching vintage data as of 2012-10-24...
Fetching vintage data as of 2012-12-12...
Fetching vintage data as of 2013-01-30...
Fetching vintage data as of 2013-03-20...
Fetching vintage data as of 2013-05-01...
Fetching vintage data as of 2013-06-19...
Fetching vintage data as of 2013-07-31...
Fetching vintage data as of 2013-09-18...
Fetching vintage data as of 2013-10-30...
Fetching vintage data as of 2013-1

In [ ]:
macro_df

,FEDFUNDS,inflation_pce,output_gap,inflation_cpi
date,,,,
2011-01-26,0.18,0.435096,2.186326,NaN
2011-03-15,0.16,0.779888,2.891245,1.592393
2011-04-27,0.14,1.083541,2.302058,2.447857
2011-06-22,0.09,1.397170,2.769962,2.945433
2011-08-09,0.07,0.822176,0.796032,2.690601
2011-09-21,0.10,0.864189,0.723113,3.332877
2011-11-02,0.08,0.773223,0.763681,3.563378
2011-12-13,0.08,0.826039,0.650487,3.431399


In [ ]:
# Test: what did GDP look like as of 2011-01-26 vs today's revised figure
vintage = get_vintage_observation('GDPC1', '2011-01-26', FRED_API_KEY)
revised = get_vintage_observation('GDPC1', '2024-01-01', FRED_API_KEY)

print("Vintage (as of 2011-01-26):")
print(vintage.tail(4))

print("\nRevised (current):")
print(revised.tail(4))

Vintage (as of 2011-01-26):


AttributeError: 'NoneType' object has no attribute 'tail'

In [ ]:
test_date = "2015-09-17"

print("--- FEDFUNDS ---")
ffr = get_vintage_observation('FEDFUNDS', test_date, FRED_API_KEY)
print(f"Length: {len(ffr) if ffr is not None else 'None'}")
print(ffr)

print("\n--- PCE ---")
pce = get_vintage_observation('PCEPI', test_date, FRED_API_KEY)
print(f"Length: {len(pce) if pce is not None else 'None'}")
print(pce)

print("\n--- GDPC1 ---")
gdp = get_vintage_observation('GDPC1', test_date, FRED_API_KEY)
print(f"Length: {len(gdp) if gdp is not None else 'None'}")
print(gdp)

print("\n--- GDPPOT ---")
pot = get_vintage_observation('GDPPOT', test_date, FRED_API_KEY)
print(f"Length: {len(pot) if pot is not None else 'None'}")
print(pot)

--- FEDFUNDS ---
Length: 24
date
2013-09-01    0.08
2013-10-01    0.09
2013-11-01    0.08
2013-12-01    0.09
2014-01-01    0.07
2014-02-01    0.07
2014-03-01    0.08
2014-04-01    0.09
2014-05-01    0.09
2014-06-01    0.10
2014-07-01    0.09
2014-08-01    0.09
2014-09-01    0.09
2014-10-01    0.09
2014-11-01    0.09
2014-12-01    0.12
2015-01-01    0.11
2015-02-01    0.11
2015-03-01    0.11
2015-04-01    0.12
2015-05-01    0.12
2015-06-01    0.13
2015-07-01    0.13
2015-08-01    0.14
Name: value, dtype: float64

--- PCE ---
Length: 24
date
2015-01-01    108.265
2015-01-01    108.272
2015-01-01    108.301
2015-01-01    108.281
2015-01-01    108.267
2015-02-01    108.776
2015-02-01    108.448
2015-02-01    108.450
2015-02-01    108.460
2015-02-01    108.486
2015-03-01    109.015
2015-03-01    108.650
2015-03-01    108.622
2015-03-01    108.646
2015-04-01    109.075
2015-04-01    109.063
2015-04-01    108.693
2015-04-01    108.649
2015-05-01    109.412
2015-05-01    109.405
2015-05-01    

In [ ]:
import pandas as pd
import requests
import datetime
import os
from google.colab import drive

# ==========================================
# 1. CONFIGURATION
# ==========================================

FRED_API_KEY = "94afca2e4faf7c6dc2cbf15deca6ce7b"
OUTPUT_PATH = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data_vintage.csv"

FOMC_DATES = [
    # 2011
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22",
    "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    # 2012
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20",
    "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    # 2013
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19",
    "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    # 2014
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18",
    "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    # 2015
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17",
    "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    # 2016
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15",
    "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    # 2017
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14",
    "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    # 2018
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13",
    "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    # 2019
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19",
    "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29",
    "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05",
    "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16",
    "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15",
    "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14",
    "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12",
    "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18",
    "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
]

# ==========================================
# 2. CORE FETCH FUNCTION
# ==========================================

def get_vintage_observation(series_id, as_of_date, api_key):
    """
    Fetches real-time vintage data for a series as it would have looked
    on a specific date. No limit — fetches all observations then deduplicates,
    keeping only the most recent revision per observation date.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id':         series_id,
        'api_key':           api_key,
        'file_type':         'json',
        'realtime_start':    '1990-01-01',
        'realtime_end':      as_of_date,
        'observation_start': '2009-01-01',
        'observation_end':   as_of_date,
        'sort_order':        'asc',
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
    except Exception as e:
        print(f"    API error for {series_id} on {as_of_date}: {e}")
        return None

    if 'observations' not in data or len(data['observations']) == 0:
        print(f"    No observations returned for {series_id} on {as_of_date}")
        return None

    observations = pd.DataFrame(data['observations'])
    observations['value'] = pd.to_numeric(observations['value'], errors='coerce')
    observations['date'] = pd.to_datetime(observations['date'])
    observations = observations.set_index('date').sort_index()

    # Keep only the most recent revision per observation date
    # This is critical — FRED returns all historical revisions stacked
    observations = observations[~observations.index.duplicated(keep='last')]

    # Drop NaN values
    observations = observations['value'].dropna()

    return observations


# ==========================================
# 3. MAIN DATA CONSTRUCTION
# ==========================================

def get_realtime_taylor_data(fomc_dates, api_key):
    """
    For each FOMC meeting date, fetches the vintage of each macro series
    as it would have been available to policymakers at that exact time.
    """
    records = []

    for meeting_date in fomc_dates:
        print(f"Fetching vintage data as of {meeting_date}...")
        record = {'date': meeting_date}

        # ---- FED FUNDS RATE ----
        ffr = get_vintage_observation('FEDFUNDS', meeting_date, api_key)
        if ffr is not None and len(ffr) >= 1:
            record['FEDFUNDS'] = ffr.iloc[-1]
        else:
            print(f"  Warning: No FEDFUNDS data for {meeting_date}")

        # ---- PCE INFLATION (YoY, vintage) ----
        pce = get_vintage_observation('PCEPI', meeting_date, api_key)
        if pce is not None and len(pce) >= 13:
            record['inflation_pce'] = (pce.iloc[-1] / pce.iloc[-13] - 1) * 100
            # Taylor Rule: deviation from 2% target
            record['inflation_pce_gap'] = record['inflation_pce'] - 2.0
        else:
            print(f"  Warning: Not enough PCE data for {meeting_date} (got {len(pce) if pce is not None else 0} obs)")

        # ---- CPI INFLATION (YoY, vintage) ----
        cpi = get_vintage_observation('CPIAUCSL', meeting_date, api_key)
        if cpi is not None and len(cpi) >= 13:
            record['inflation_cpi'] = (cpi.iloc[-1] / cpi.iloc[-13] - 1) * 100
        else:
            print(f"  Warning: Not enough CPI data for {meeting_date} (got {len(cpi) if cpi is not None else 0} obs)")

        # ---- OUTPUT GAP (vintage GDP, with sanity check) ----
        gdp = get_vintage_observation('GDPC1', meeting_date, api_key)
        pot = get_vintage_observation('GDPPOT', meeting_date, api_key)

        if gdp is not None and pot is not None and len(gdp) >= 1 and len(pot) >= 1:
            latest_gdp = gdp.iloc[-1]
            latest_pot = pot.iloc[-1]
            gap = 100 * (latest_gdp - latest_pot) / latest_pot

            # Sanity check — output gap should never exceed ±15%
            if -15 < gap < 15:
                record['output_gap'] = gap
            else:
                print(f"  Warning: Implausible output gap {gap:.1f}% for {meeting_date} — skipping")
        else:
            print(f"  Warning: Missing GDP/GDPPOT data for {meeting_date}")

        records.append(record)

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

    return df


# ==========================================
# 4. EXECUTION
# ==========================================

if __name__ == "__main__":

    # Mount Google Drive
    drive.mount('/content/drive')

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Fetch data
    print("Starting real-time vintage data fetch...\n")
    macro_df = get_realtime_taylor_data(FOMC_DATES, FRED_API_KEY)

    # Summary
    print("\n--- DATA SUMMARY ---")
    print(f"Total FOMC dates:     {len(macro_df)}")
    print(f"Complete rows:        {macro_df.dropna().shape[0]}")
    print(f"Columns:              {macro_df.columns.tolist()}")
    print("\n--- FIRST 5 ROWS ---")
    print(macro_df.head())
    print("\n--- MISSING VALUES PER COLUMN ---")
    print(macro_df.isnull().sum())

    # Save
    macro_df.to_csv(OUTPUT_PATH)
    print(f"\nSUCCESS — saved to: {OUTPUT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting real-time vintage data fetch...

Fetching vintage data as of 2011-01-26...
Fetching vintage data as of 2011-03-15...
Fetching vintage data as of 2011-04-27...
Fetching vintage data as of 2011-06-22...
Fetching vintage data as of 2011-08-09...
Fetching vintage data as of 2011-09-21...
Fetching vintage data as of 2011-11-02...
Fetching vintage data as of 2011-12-13...
Fetching vintage data as of 2012-01-25...
Fetching vintage data as of 2012-03-13...
Fetching vintage data as of 2012-04-25...
Fetching vintage data as of 2012-06-20...
Fetching vintage data as of 2012-08-01...
Fetching vintage data as of 2012-09-13...
Fetching vintage data as of 2012-10-24...
Fetching vintage data as of 2012-12-12...
Fetching vintage data as of 2013-01-30...
Fetching vintage data as of 2013-03-20...
Fetching vintage data as of 2013-05-01...
Fetching vintage data as of 201

In [ ]:
import pandas as pd
import requests
import datetime
import os
import time
from google.colab import drive

# ==========================================
# 1. CONFIGURATION
# ==========================================

FRED_API_KEY = "94afca2e4faf7c6dc2cbf15deca6ce7b" # Note: It's usually best practice to keep API keys hidden, but I've left it here so your script runs immediately!
OUTPUT_PATH = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data_vintage.csv"

FOMC_DATES = [
    # 2011
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22",
    "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    # 2012
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20",
    "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    # 2013
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19",
    "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    # 2014
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18",
    "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    # 2015
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17",
    "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    # 2016
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15",
    "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    # 2017
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14",
    "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    # 2018
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13",
    "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    # 2019
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19",
    "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29",
    "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05",
    "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16",
    "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15",
    "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14",
    "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12",
    "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18",
    "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
]

# ==========================================
# 2. CORE FETCH FUNCTION
# ==========================================

def get_vintage_observation(series_id, as_of_date, api_key, max_retries=3):
    """
    Fetches real-time vintage data for a series as it would have looked
    on a specific date, with rate-limit handling and backoff.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id':         series_id,
        'api_key':           api_key,
        'file_type':         'json',
        'realtime_start':    '1990-01-01',
        'realtime_end':      as_of_date,
        'observation_start': '2009-01-01',
        'observation_end':   as_of_date,
        'sort_order':        'asc',
    }

    data = None
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params)

            # Handle rate limiting specifically
            if response.status_code == 429:
                print(f"    [!] Rate limited by FRED. Waiting 10s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(10)
                continue

            # Raise an error for any other bad HTTP status (400, 500, etc.)
            response.raise_for_status()

            data = response.json()
            break # Success, exit the retry loop

        except requests.exceptions.RequestException as e:
            print(f"    [!] API error for {series_id} on {as_of_date}: {e}")
            if attempt == max_retries - 1:
                return None
            time.sleep(2)

    # Respect the 120 requests/minute limit (~2 requests/second)
    time.sleep(0.6)

    if not data or 'observations' not in data or len(data['observations']) == 0:
        print(f"    No observations returned for {series_id} on {as_of_date}")
        return None

    observations = pd.DataFrame(data['observations'])
    observations['value'] = pd.to_numeric(observations['value'], errors='coerce')
    observations['date'] = pd.to_datetime(observations['date'])
    observations = observations.set_index('date').sort_index()

    # Keep only the most recent revision per observation date
    # This is critical — FRED returns all historical revisions stacked
    observations = observations[~observations.index.duplicated(keep='last')]

    # Drop NaN values
    observations = observations['value'].dropna()

    return observations


# ==========================================
# 3. MAIN DATA CONSTRUCTION
# ==========================================

def get_realtime_taylor_data(fomc_dates, api_key):
    """
    For each FOMC meeting date, fetches the vintage of each macro series
    as it would have been available to policymakers at that exact time.
    """
    records = []

    for meeting_date in fomc_dates:
        print(f"Fetching vintage data as of {meeting_date}...")
        record = {'date': meeting_date}

        # ---- FED FUNDS RATE ----
        ffr = get_vintage_observation('FEDFUNDS', meeting_date, api_key)
        if ffr is not None and len(ffr) >= 1:
            record['FEDFUNDS'] = ffr.iloc[-1]
        else:
            print(f"  Warning: No FEDFUNDS data for {meeting_date}")

        # ---- PCE INFLATION (YoY, vintage) ----
        pce = get_vintage_observation('PCEPI', meeting_date, api_key)
        if pce is not None and len(pce) >= 13:
            record['inflation_pce'] = (pce.iloc[-1] / pce.iloc[-13] - 1) * 100
            # Taylor Rule: deviation from 2% target
            record['inflation_pce_gap'] = record['inflation_pce'] - 2.0
        else:
            print(f"  Warning: Not enough PCE data for {meeting_date} (got {len(pce) if pce is not None else 0} obs)")

        # ---- CPI INFLATION (YoY, vintage) ----
        cpi = get_vintage_observation('CPIAUCSL', meeting_date, api_key)
        if cpi is not None and len(cpi) >= 13:
            record['inflation_cpi'] = (cpi.iloc[-1] / cpi.iloc[-13] - 1) * 100
        else:
            print(f"  Warning: Not enough CPI data for {meeting_date} (got {len(cpi) if cpi is not None else 0} obs)")

        # ---- OUTPUT GAP (vintage GDP, with sanity check) ----
        gdp = get_vintage_observation('GDPC1', meeting_date, api_key)
        pot = get_vintage_observation('GDPPOT', meeting_date, api_key)

        if gdp is not None and pot is not None and len(gdp) >= 1 and len(pot) >= 1:
            latest_gdp = gdp.iloc[-1]
            latest_pot = pot.iloc[-1]
            gap = 100 * (latest_gdp - latest_pot) / latest_pot

            # Sanity check — output gap should never exceed ±15%
            if -15 < gap < 15:
                record['output_gap'] = gap
            else:
                print(f"  Warning: Implausible output gap {gap:.1f}% for {meeting_date} — skipping")
        else:
            print(f"  Warning: Missing GDP/GDPPOT data for {meeting_date}")

        records.append(record)

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

    return df


# ==========================================
# 4. EXECUTION
# ==========================================

if __name__ == "__main__":

    # Mount Google Drive
    drive.mount('/content/drive')

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Fetch data
    print("Starting real-time vintage data fetch...\n")
    macro_df = get_realtime_taylor_data(FOMC_DATES, FRED_API_KEY)

    # Summary
    print("\n--- DATA SUMMARY ---")
    print(f"Total FOMC dates:     {len(macro_df)}")
    print(f"Complete rows:        {macro_df.dropna().shape[0]}")
    print(f"Columns:              {macro_df.columns.tolist()}")
    print("\n--- FIRST 5 ROWS ---")
    print(macro_df.head())
    print("\n--- MISSING VALUES PER COLUMN ---")
    print(macro_df.isnull().sum())

    # Save
    macro_df.to_csv(OUTPUT_PATH)
    print(f"\nSUCCESS — saved to: {OUTPUT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting real-time vintage data fetch...

Fetching vintage data as of 2011-01-26...
Fetching vintage data as of 2011-03-15...
Fetching vintage data as of 2011-04-27...
    [!] API error for GDPC1 on 2011-04-27: 500 Server Error: Internal Server Error for url: https://api.stlouisfed.org/fred/series/observations?series_id=GDPC1&api_key=94afca2e4faf7c6dc2cbf15deca6ce7b&file_type=json&realtime_start=1990-01-01&realtime_end=2011-04-27&observation_start=2009-01-01&observation_end=2011-04-27&sort_order=asc
    [!] API error for GDPC1 on 2011-04-27: 500 Server Error: Internal Server Error for url: https://api.stlouisfed.org/fred/series/observations?series_id=GDPC1&api_key=94afca2e4faf7c6dc2cbf15deca6ce7b&file_type=json&realtime_start=1990-01-01&realtime_end=2011-04-27&observation_start=2009-01-01&observation_end=2011-04-27&sort_order=asc
Fetching vintage data as of 2

In [ ]:
import requests

FRED_API_KEY = "94afca2e4faf7c6dc2cbf15deca6ce7b"

def run_sanity_check(series_id, observation_date, fomc_meeting_date):
    """
    Compares the value of a macroeconomic series as it was reported on the
    FOMC meeting date vs. how it is reported today (after revisions).
    """
    url = "https://api.stlouisfed.org/fred/series/observations"

    # 1. VINTAGE DATA (What the FOMC saw on meeting day)
    params_vintage = {
        'series_id': series_id,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'observation_start': observation_date,
        'observation_end': observation_date,
        'realtime_end': fomc_meeting_date  # The magic ALFRED parameter!
    }
    res_vintage = requests.get(url, params=params_vintage).json()
    val_vintage = res_vintage['observations'][-1]['value'] if 'observations' in res_vintage and res_vintage['observations'] else "N/A"

    # 2. CURRENT DATA (What we know today)
    params_current = {
        'series_id': series_id,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'observation_start': observation_date,
        'observation_end': observation_date
        # Leaving realtime_end blank defaults to today's fully revised data
    }
    res_current = requests.get(url, params=params_current).json()
    val_current = res_current['observations'][-1]['value'] if 'observations' in res_current and res_current['observations'] else "N/A"

    # 3. PRINT RESULTS
    print(f"--- Sanity Check: {series_id} for observation {observation_date} ---")
    print(f"Vintage Value (as known on {fomc_meeting_date}): {val_vintage}")
    print(f"Current Value (fully revised as of today) : {val_current}")
    print("-" * 65)

# --- LET'S TEST SOME FAMOUS REVISIONS ---

# Test 1: Real GDP (GDPC1) for Q4 2008 (Height of the Financial Crisis)
# We check what the Fed thought Q4 2008 GDP was during their April 2009 meeting vs today.
run_sanity_check('GDPC1', '2008-10-01', '2009-04-29')

# Test 2: Core PCE (PCEPI) for January 2012
# We check what the Fed thought January 2012 PCE was during their March 2012 meeting vs today.
run_sanity_check('PCEPI', '2012-01-01', '2012-03-13')

# Test 3: Unemployment Rate (UNRATE) for September 2008
# We check what the Fed thought Sep 2008 unemployment was during their Dec 2008 meeting vs today.
run_sanity_check('UNRATE', '2008-09-01', '2008-12-16')

--- Sanity Check: GDPC1 for observation 2008-10-01 ---
Vintage Value (as known on 2009-04-29): 11522.1
Current Value (fully revised as of today) : 16485.35
-----------------------------------------------------------------
--- Sanity Check: PCEPI for observation 2012-01-01 ---
Vintage Value (as known on 2012-03-13): 114.954
Current Value (fully revised as of today) : 93.894
-----------------------------------------------------------------
--- Sanity Check: UNRATE for observation 2008-09-01 ---
Vintage Value (as known on 2008-12-16): 6.1
Current Value (fully revised as of today) : 6.1
-----------------------------------------------------------------


121


In [ ]:
import pandas as pd
import requests
import datetime
import os
import time
from google.colab import drive

# ==========================================
# 1. CONFIGURATION
# ==========================================

FRED_API_KEY = "94afca2e4faf7c6dc2cbf15deca6ce7b"
OUTPUT_PATH = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data_vintage_2.csv"

FOMC_DATES = [
    # 2011
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22",
    "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    # 2012
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20",
    "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    # 2013
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19",
    "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    # 2014
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18",
    "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    # 2015
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17",
    "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    # 2016
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15",
    "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    # 2017
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14",
    "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    # 2018
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13",
    "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    # 2019
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19",
    "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29",
    "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05",
    "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-9",
    "2021-07-28", "2021-09-15", "2021-10-", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15",
    "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14",
    "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12",
    "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18",
    "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
]

# ==========================================
# 2. CORE FETCH FUNCTION
# ==========================================

def get_vintage_observation(series_id, as_of_date, api_key, max_retries=3):
    """
    Fetches real-time vintage data for a series as it would have looked
    on a specific date, with rate-limit handling and backoff.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id':         series_id,
        'api_key':           api_key,
        'file_type':         'json',
        'realtime_start':    '1990-01-01',
        'realtime_end':      as_of_date,
        'observation_start': '2009-01-01',
        'observation_end':   as_of_date,
        'sort_order':        'asc',
    }

    data = None
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params)

            # Handle rate limiting specifically
            if response.status_code == 429:
                print(f"    [!] Rate limited by FRED. Waiting 10s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(10)
                continue

            # Raise an error for any other bad HTTP status (400, 500, etc.)
            response.raise_for_status()

            data = response.json()
            break # Success, exit the retry loop

        except requests.exceptions.RequestException as e:
            print(f"    [!] API error for {series_id} on {as_of_date}: {e}")
            if attempt == max_retries - 1:
                return None
            time.sleep(2)

    # Respect the 120 requests/minute limit (~2 requests/second)
    time.sleep(0.6)

    if not data or 'observations' not in data or len(data['observations']) == 0:
        print(f"    No observations returned for {series_id} on {as_of_date}")
        return None

    observations = pd.DataFrame(data['observations'])
    observations['value'] = pd.to_numeric(observations['value'], errors='coerce')
    observations['date'] = pd.to_datetime(observations['date'])
    observations = observations.set_index('date').sort_index()

    # Keep only the most recent revision per observation date
    # This is critical — FRED returns all historical revisions stacked
    observations = observations[~observations.index.duplicated(keep='last')]

    # Drop NaN values
    observations = observations['value'].dropna()

    return observations


# ==========================================
# 3. MAIN DATA CONSTRUCTION
# ==========================================

def get_realtime_taylor_data(fomc_dates, api_key):
    """
    For each FOMC meeting date, fetches the vintage of each macro series
    as it would have been available to policymakers at that exact time.
    """
    records = []

    for meeting_date in fomc_dates:
        print(f"Fetching vintage data as of {meeting_date}...")
        record = {'date': meeting_date}

        # ---- FED FUNDS RATE ----
        ffr = get_vintage_observation('FEDFUNDS', meeting_date, api_key)
        if ffr is not None and len(ffr) >= 1:
            record['FEDFUNDS'] = ffr.iloc[-1]
        else:
            print(f"  Warning: No FEDFUNDS data for {meeting_date}")

        # ---- PCE INFLATION (YoY, vintage) ----
        pce = get_vintage_observation('PCEPI', meeting_date, api_key)
        if pce is not None and len(pce) >= 13:
            record['inflation_pce'] = (pce.iloc[-1] / pce.iloc[-13] - 1) * 100
            # Taylor Rule: deviation from 2% target
            record['inflation_pce_gap'] = record['inflation_pce'] - 2.0
        else:
            print(f"  Warning: Not enough PCE data for {meeting_date} (got {len(pce) if pce is not None else 0} obs)")

        # ---- CPI INFLATION (YoY, vintage) ----
        cpi = get_vintage_observation('CPIAUCSL', meeting_date, api_key)
        if cpi is not None and len(cpi) >= 13:
            record['inflation_cpi'] = (cpi.iloc[-1] / cpi.iloc[-13] - 1) * 100
        else:
            print(f"  Warning: Not enough CPI data for {meeting_date} (got {len(cpi) if cpi is not None else 0} obs)")

        # ---- UNEMPLOYMENT GAP (vintage UNRATE vs NROU) ----
        unrate = get_vintage_observation('UNRATE', meeting_date, api_key)
        nrou = get_vintage_observation('NROU', meeting_date, api_key)

        if unrate is not None and nrou is not None and len(unrate) >= 1 and len(nrou) >= 1:
            latest_unrate = unrate.iloc[-1]
            latest_nrou = nrou.iloc[-1]

            # Unemployment gap = Current Unemployment - Natural Rate
            gap = latest_unrate - latest_nrou

            # Sanity check — unemployment gap shouldn't exceed extreme historical levels (e.g., ±15%)
            if -15 < gap < 15:
                record['unemployment_gap'] = gap
            else:
                print(f"  Warning: Implausible unemployment gap {gap:.1f}% for {meeting_date} — skipping")
        else:
            print(f"  Warning: Missing UNRATE/NROU data for {meeting_date}")

        records.append(record)

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

    return df


# ==========================================
# 4. EXECUTION
# ==========================================

if __name__ == "__main__":

    # Mount Google Drive
    drive.mount('/content/drive')

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Fetch data
    print("Starting real-time vintage data fetch...\n")
    macro_df = get_realtime_taylor_data(FOMC_DATES, FRED_API_KEY)

    # Summary
    print("\n--- DATA SUMMARY ---")
    print(f"Total FOMC dates:     {len(macro_df)}")
    print(f"Complete rows:        {macro_df.dropna().shape[0]}")
    print(f"Columns:              {macro_df.columns.tolist()}")
    print("\n--- FIRST 5 ROWS ---")
    print(macro_df.head())
    print("\n--- MISSING VALUES PER COLUMN ---")
    print(macro_df.isnull().sum())

    # Save
    macro_df.to_csv(OUTPUT_PATH)
    print(f"\nSUCCESS — saved to: {OUTPUT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting real-time vintage data fetch...

Fetching vintage data as of 2011-01-26...
    [!] API error for NROU on 2011-01-26: 400 Client Error: Bad Request for url: https://api.stlouisfed.org/fred/series/observations?series_id=NROU&api_key=94afca2e4faf7c6dc2cbf15deca6ce7b&file_type=json&realtime_start=1990-01-01&realtime_end=2011-01-26&observation_start=2009-01-01&observation_end=2011-01-26&sort_order=asc
    [!] API error for NROU on 2011-01-26: 400 Client Error: Bad Request for url: https://api.stlouisfed.org/fred/series/observations?series_id=NROU&api_key=94afca2e4faf7c6dc2cbf15deca6ce7b&file_type=json&realtime_start=1990-01-01&realtime_end=2011-01-26&observation_start=2009-01-01&observation_end=2011-01-26&sort_order=asc
    [!] API error for NROU on 2011-01-26: 400 Client Error: Bad Request for url: https://api.stlouisfed.org/fred/series/observations?seri

In [14]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re
from google.colab import drive

# 1. MOUNT DRIVE
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURATION ---
BASE_ROOT    = "/content/drive/MyDrive/HEC Thesis/Text Data/fomc_statements"
BASE_URL     = "https://www.federalreserve.gov"
CALENDAR_URL = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"
TARGET_YEARS = [2021, 2022, 2023, 2024, 2025]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

os.makedirs(BASE_ROOT, exist_ok=True)

# STRICTLY the main statement (a.htm), ignoring Implementation Notes (a1.htm)
STATEMENT_PATTERN = re.compile(r'/newsevents/pressreleases/monetary(\d{8})a\.htm', re.IGNORECASE)

def extract_statement_text(soup):
    """Pull clean statement text from a Fed press release page."""
    for selector in ['#article', '.col-xs-12.col-sm-8.col-sm-push-0']:
        block = soup.select_one(selector)
        if block:
            return block.get_text(separator='\n').strip()
    return soup.get_text(separator='\n').strip()

def download_all_statements():
    session = requests.Session()
    print(f"--- Fetching Main Calendar: {CALENDAR_URL} ---")

    res  = session.get(CALENDAR_URL, headers=HEADERS)
    soup = BeautifulSoup(res.text, 'html.parser')

    statement_links = []
    for a in soup.find_all('a', href=True):
        href  = a['href']
        match = STATEMENT_PATTERN.search(href)
        if match:
            date_str = match.group(1)
            year     = int(date_str[:4])
            if year in TARGET_YEARS:
                full_url = urljoin(BASE_URL, href)
                if full_url not in [link[1] for link in statement_links]:
                    statement_links.append((date_str, full_url))

    # Sort chronologically
    statement_links.sort(key=lambda x: x[0])
    print(f"Found {len(statement_links)} statements to download for {TARGET_YEARS}.\n")

    for date_str, stmt_url in statement_links:
        save_path = os.path.join(BASE_ROOT, f"statement_{date_str}.txt")

        try:
            print(f"  -> Downloading: {date_str}...")
            stmt_res  = session.get(stmt_url, headers=HEADERS)
            stmt_soup = BeautifulSoup(stmt_res.text, 'html.parser')
            text      = extract_statement_text(stmt_soup)

            # Unconditionally write the file
            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(f"SOURCE: {stmt_url}\nDATE: {date_str}\n\n")
                f.write(text)

            time.sleep(1) # Be nice to the Fed's servers

        except Exception as e:
            print(f"  ❌ Error on {date_str}: {e}")

    print("\n--- Download Complete! ---")

if __name__ == "__main__":
    download_all_statements()

--- Fetching Main Calendar: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm ---
Found 41 statements to download for [2021, 2022, 2023, 2024, 2025].

  -> Downloading: 20210127...
  -> Downloading: 20210317...
  -> Downloading: 20210428...
  -> Downloading: 20210616...
  -> Downloading: 20210728...
  -> Downloading: 20210922...
  -> Downloading: 20211103...
  -> Downloading: 20211215...
  -> Downloading: 20220126...
  -> Downloading: 20220316...
  -> Downloading: 20220504...
  -> Downloading: 20220615...
  -> Downloading: 20220727...
  -> Downloading: 20220921...
  -> Downloading: 20221102...
  -> Downloading: 20221214...
  -> Downloading: 20230201...
  -> Downloading: 20230322...
  -> Downloading: 20230503...
  -> Downloading: 20230614...
  -> Downloading: 20230726...
  -> Downloading: 20230920...
  -> Downloading: 20231101...
  -> Downloading: 20231213...
  -> Downloading: 20240131...
  -> Downloading: 20240320...
  -> Downloading: 20240501...
  -> Downloading: 2024061

In [8]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re
from google.colab import drive

# 1. MOUNT DRIVE
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURATION ---
BASE_URL     = "https://www.federalreserve.gov"
DOWNLOAD_DIR = "/content/drive/MyDrive/HEC Thesis/fomc_statements"
TARGET_YEARS = ["2011", "2012", "2013", "2014", "2015",
                "2016", "2017", "2018", "2019", "2020"]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# FIXED: actual Fed statement URLs end in 'a.htm', not 'a1.htm'
# e.g. /newsevents/pressreleases/monetary20181219a.htm
STATEMENT_PATTERN = re.compile(
    r'/newsevents/pressreleases/monetary(\d{8})a\.htm', re.IGNORECASE
)

# Historical year pages follow a predictable URL:
# https://www.federalreserve.gov/monetarypolicy/fomchistorical{YEAR}.htm
def get_year_url(year):
    return f"{BASE_URL}/monetarypolicy/fomchistorical{year}.htm"


def extract_statement_text(soup):
    for selector in ['#article', '.col-xs-12.col-sm-8']:
        block = soup.select_one(selector)
        if block:
            return block.get_text(separator='\n').strip()
    return soup.get_text(separator='\n').strip()


def run_statement_scraper_historical():
    session = requests.Session()

    for year in sorted(TARGET_YEARS):
        year_url = get_year_url(year)
        print(f"\n=== Processing Year: {year} ({year_url}) ===")

        try:
            year_res  = session.get(year_url, headers=HEADERS)
            year_soup = BeautifulSoup(year_res.text, 'html.parser')

            # Match all statement links by URL pattern (link text is irrelevant)
            statement_links = []
            for a in year_soup.find_all('a', href=True):
                match = STATEMENT_PATTERN.search(a['href'])
                if match:
                    date_str = match.group(1)
                    full_url = urljoin(BASE_URL, a['href'])
                    if (date_str, full_url) not in statement_links:
                        statement_links.append((date_str, full_url))

            statement_links.sort(key=lambda x: x[0])
            print(f"  Found {len(statement_links)} statements.")

            for date_str, stmt_url in statement_links:
                save_path = os.path.join(DOWNLOAD_DIR, f"statement_{date_str}.txt")
                if os.path.exists(save_path):
                    print(f"    -> Already exists: {date_str}")
                    continue
                try:
                    print(f"    -> Downloading: {date_str}")
                    stmt_res  = session.get(stmt_url, headers=HEADERS)
                    stmt_soup = BeautifulSoup(stmt_res.text, 'html.parser')
                    text      = extract_statement_text(stmt_soup)
                    with open(save_path, 'w', encoding='utf-8') as f:
                        f.write(f"SOURCE: {stmt_url}\nDATE: {date_str}\n\n{text}")
                    time.sleep(1)
                except Exception as e:
                    print(f"    Error on {date_str}: {e}")

        except Exception as e:
            print(f"  Error accessing {year_url}: {e}")

    print(f"\n--- Done! Historical statements saved to fomc_statements/ ---")


if __name__ == "__main__":
    run_statement_scraper_historical()


=== Processing Year: 2011 (https://www.federalreserve.gov/monetarypolicy/fomchistorical2011.htm) ===
  Found 8 statements.
    -> Downloading: 20110126
    -> Downloading: 20110315
    -> Downloading: 20110427
    -> Downloading: 20110622
    -> Downloading: 20110809
    -> Downloading: 20110921
    -> Downloading: 20111102
    -> Downloading: 20111213

=== Processing Year: 2012 (https://www.federalreserve.gov/monetarypolicy/fomchistorical2012.htm) ===
  Found 8 statements.
    -> Downloading: 20120125
    -> Downloading: 20120313
    -> Downloading: 20120425
    -> Downloading: 20120620
    -> Downloading: 20120801
    -> Downloading: 20120913
    -> Downloading: 20121024
    -> Downloading: 20121212

=== Processing Year: 2013 (https://www.federalreserve.gov/monetarypolicy/fomchistorical2013.htm) ===
  Found 8 statements.
    -> Downloading: 20130130
    -> Downloading: 20130320
    -> Downloading: 20130501
    -> Downloading: 20130619
    -> Downloading: 20130731
    -> Downloading:

In [16]:
import os
import json
import re

# --- CONFIGURATION ---
STATEMENTS_DIR = "/content/drive/MyDrive/HEC Thesis/Text Data/fomc_statements"
OUTPUT_DIR     = "/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dates to exclude: unscheduled/technical meetings with no policy stance change
EXCLUDE_DATES = {

}


def clean_statement_text(text):
    """Remove boilerplate headers/footers scraped from the Fed page."""
    # Remove the SOURCE/DATE header lines we wrote during scraping
    text = re.sub(r'^SOURCE:.*\n', '', text)
    text = re.sub(r'^DATE:.*\n', '', text)

    # Remove common Fed page chrome
    boilerplate = [
        r"Federal Reserve issues FOMC statement",
        r"For release at \d+:\d+ [ap]\.m\. [A-Z]+",
        r"For immediate release",
        r"Last Update:.*",
        r"Board of Governors of the Federal Reserve System",
        r"20th Street and Constitution Avenue.*",
        r"Home\s*›.*",           # breadcrumb nav
        r"Share\s*Twitter.*",    # social share buttons
        r"Back to Top",
        r"Stay Connected.*",
        r"Subscribe to.*",
    ]
    for pattern in boilerplate:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # Collapse excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def process_statements():
    txt_files = sorted([
        f for f in os.listdir(STATEMENTS_DIR)
        if f.endswith('.txt') and f.startswith('statement_')
    ])

    print(f"Found {len(txt_files)} statement files.")

    skipped   = []
    processed = []
    errors    = []

    for filename in txt_files:
        # Extract date from filename: statement_YYYYMMDD.txt
        date_match = re.search(r'(\d{8})', filename)
        if not date_match:
            print(f"  [!] Could not parse date from {filename}, skipping.")
            errors.append(filename)
            continue

        date_str = date_match.group(1)

        # Skip excluded meetings
        if date_str in EXCLUDE_DATES:
            print(f"  -> Skipping excluded date: {date_str}")
            skipped.append(date_str)
            continue

        filepath    = os.path.join(STATEMENTS_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, f"statement_{date_str}.json")

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                raw_text = f.read()

            cleaned_text = clean_statement_text(raw_text)

            # Format date as YYYY-MM-DD for readability
            date_formatted = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"

            entry = {
                "source":     filename,
                "date":       date_formatted,
                "date_raw":   date_str,
                "type":       "fomc_statement",
                "text":       cleaned_text,
                "word_count": len(cleaned_text.split())
            }

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(entry, f, indent=4, ensure_ascii=False)

            processed.append(date_str)
            print(f"  ✅ {date_str}  ({entry['word_count']} words)")

        except Exception as e:
            print(f"  ❌ Error on {filename}: {e}")
            errors.append(filename)

    print(f"\n--- Summary ---")
    print(f"  Processed : {len(processed)}")
    print(f"  Skipped   : {len(skipped)}  {skipped}")
    print(f"  Errors    : {len(errors)}")
    print(f"\nJSONs saved to: {OUTPUT_DIR}")

    # Sanity check: warn if word count looks wrong for any file
    print("\n--- Word count sanity check (statements should be ~300-600 words) ---")
    for filename in sorted(os.listdir(OUTPUT_DIR)):
        if filename.endswith('.json'):
            with open(os.path.join(OUTPUT_DIR, filename)) as f:
                data = json.load(f)
            wc = data['word_count']
            flag = "⚠️  SUSPICIOUS" if wc < 100 or wc > 1500 else ""
            print(f"  {data['date']}  {wc} words  {flag}")


if __name__ == "__main__":
    process_statements()

Found 122 statement files.
  ✅ 20110126  (440 words)
  ✅ 20110315  (466 words)
  ✅ 20110427  (467 words)
  ✅ 20110622  (459 words)
  ✅ 20110809  (527 words)
  ✅ 20110921  (596 words)
  ✅ 20111102  (489 words)
  ✅ 20111213  (433 words)
  ✅ 20120125  (422 words)
  ✅ 20120313  (433 words)
  ✅ 20120425  (442 words)
  ✅ 20120620  (506 words)
  ✅ 20120801  (461 words)
  ✅ 20120913  (565 words)
  ✅ 20121024  (552 words)
  ✅ 20121212  (690 words)
  ✅ 20130130  (649 words)
  ✅ 20130320  (647 words)
  ✅ 20130501  (673 words)
  ✅ 20130619  (701 words)
  ✅ 20130731  (704 words)
  ✅ 20130918  (802 words)
  ✅ 20131030  (780 words)
  ✅ 20131218  (881 words)
  ✅ 20140129  (844 words)
  ✅ 20140319  (889 words)
  ✅ 20140430  (823 words)
  ✅ 20140618  (810 words)
  ✅ 20140730  (863 words)
  ✅ 20140917  (907 words)
  ✅ 20141029  (729 words)
  ✅ 20141217  (736 words)
  ✅ 20150128  (571 words)
  ✅ 20150318  (586 words)
  ✅ 20150429  (562 words)
  ✅ 20150617  (547 words)
  ✅ 20150729  (541 words)
  ✅ 2015091

In [5]:
import pandas as pd
import requests
import datetime
import os
import json
import time

# --- CONFIGURATION ---
FRED_API_KEY  = "727c937f8777a23ffb1fd28e68723776" # Remember to replace your leaked key!
JSON_DIR      = "/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements"
OUTPUT_PATH   = "/content/drive/MyDrive/HEC Thesis/Taylor Rule Data/taylor_rule_data_vintage.csv"


# ==========================================
# 1. LOAD FOMC DATES FROM JSON FILES
# ==========================================
def load_fomc_dates_from_json(json_dir):
    dates = []
    for filename in sorted(os.listdir(json_dir)):
        if not filename.endswith('.json'):
            continue
        filepath = os.path.join(json_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            date_str = data.get('date') or data.get('date_raw')
            if date_str:
                if len(date_str) == 8 and '-' not in date_str:
                    date_str = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
                dates.append(date_str)
        except Exception as e:
            print(f"  Warning: Could not read {filename}: {e}")

    dates = sorted(set(dates))
    print(f"✅ Loaded {len(dates)} FOMC meeting dates from JSON files.")
    return dates


# ==========================================
# 2. VINTAGE DATA FETCHER
# ==========================================
def get_vintage_observation(series_id, as_of_date, api_key):
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
            'series_id':         series_id,
            'api_key':           api_key,
            'file_type':         'json',

            # THE TIME MACHINE: Must be identical to get a single snapshot in time
            'realtime_start':    as_of_date,
            'realtime_end':      as_of_date,

            # THE DATA RUNWAY: Gives you history dating back to 2000 to calculate YoY
            'observation_start': '2000-01-01',
            'observation_end':   as_of_date,

            'sort_order':        'asc'
    }

    try:
        # Fixed: Added a slight delay to respect FRED's 120 req/min limit
        time.sleep(0.6)

        response = requests.get(url, params=params, timeout=30)

        if response.status_code != 200:
            print(f"    API HTTP {response.status_code} for {series_id}. Check rate limits.")
            return None

        data = response.json()
    except Exception as e:
        print(f"    API error for {series_id} on {as_of_date}: {e}")
        return None

    if 'observations' not in data or len(data['observations']) == 0:
        return None

    df = pd.DataFrame(data['observations'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['date']  = pd.to_datetime(df['date'])
    df = df.dropna(subset=['value']).set_index('date').sort_index()

    return df['value']


def get_latest_as_of(series, meeting_date):
    cutoff = pd.to_datetime(meeting_date)
    filtered = series[series.index <= cutoff]
    if len(filtered) == 0:
        return None, None
    return filtered.iloc[-1], filtered.index[-1]


# ==========================================
# 3. MAIN DATA COLLECTION LOOP
# ==========================================
def get_realtime_taylor_data(fomc_dates, api_key):
    records = []

    for meeting_date in fomc_dates:
        print(f"\nFetching vintage data as of {meeting_date}...")
        record = {'date': meeting_date}

        # ── Fed Funds Rate ──────────────────────────────────────────────
        ffr = get_vintage_observation('FEDFUNDS', meeting_date, api_key)
        if ffr is not None:
            val, obs_date = get_latest_as_of(ffr, meeting_date)
            if val is not None:
                record['FEDFUNDS'] = val
                print(f"  FEDFUNDS={val:.2f}% (obs: {obs_date.date()})")

        # ── PCE Inflation (YoY) ─────────────────────────────────────────
        pce = get_vintage_observation('PCEPI', meeting_date, api_key)
        if pce is not None and len(pce) >= 13:
            pce_filtered = pce[pce.index <= pd.to_datetime(meeting_date)]
            if len(pce_filtered) >= 13:
                record['inflation_pce'] = pce_filtered.pct_change(periods=12).iloc[-1] * 100

        # ── CPI Inflation (YoY) ─────────────────────────────────────────
        cpi = get_vintage_observation('CPIAUCSL', meeting_date, api_key)
        if cpi is not None and len(cpi) >= 13:
            cpi_filtered = cpi[cpi.index <= pd.to_datetime(meeting_date)]
            if len(cpi_filtered) >= 13:
                record['inflation_cpi'] = cpi_filtered.pct_change(periods=12).iloc[-1] * 100

        # ── Output Gap ──────────────────────────────────────────────────
        gdp = get_vintage_observation('GDPC1',  meeting_date, api_key)
        pot = get_vintage_observation('GDPPOT', meeting_date, api_key)

        if gdp is not None and pot is not None:
            gdp_val, gdp_date = get_latest_as_of(gdp, meeting_date)
            pot_val, pot_date = get_latest_as_of(pot, meeting_date)

            if gdp_val is not None and pot_val is not None:
                output_gap = 100 * (gdp_val - pot_val) / pot_val
                record['output_gap'] = output_gap
                print(f"  Output gap={output_gap:.2f}% (GDP obs: {gdp_date.date()}, POT obs: {pot_date.date()})")

        # ── Unemployment Gap ────────────────────────────────────────────
        unrate = get_vintage_observation('UNRATE', meeting_date, api_key)
        nrou   = get_vintage_observation('NROU',   meeting_date, api_key)

        if unrate is not None and nrou is not None:
            ur_val,   ur_date   = get_latest_as_of(unrate, meeting_date)
            nrou_val, nrou_date = get_latest_as_of(nrou,   meeting_date)

            if ur_val is not None and nrou_val is not None:
                unemp_gap = ur_val - nrou_val
                record['unemployment_gap'] = unemp_gap
                print(f"  Unemployment gap={unemp_gap:.2f}% (UNRATE obs: {ur_date.date()}, NROU obs: {nrou_date.date()})")

        records.append(record)

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').set_index('date')
    return df


# ==========================================
# 4. RUN
# ==========================================
if __name__ == "__main__":
    fomc_dates = load_fomc_dates_from_json(JSON_DIR)

    if fomc_dates:
        print(f"\nDate range: {fomc_dates[0]} → {fomc_dates[-1]}")
        print(f"Total meetings: {len(fomc_dates)}\n")

        macro_df = get_realtime_taylor_data(fomc_dates, FRED_API_KEY)

        print("\n--- Data Summary ---")
        print(macro_df.describe().round(2))

        macro_df.to_csv(OUTPUT_PATH)
        print(f"\n✅ Saved to {OUTPUT_PATH}")

✅ Loaded 121 FOMC meeting dates from JSON files.

Date range: 2011-01-26 → 2025-12-10
Total meetings: 121


Fetching vintage data as of 2011-01-26...
  FEDFUNDS=0.18% (obs: 2010-12-01)
  Output gap=-6.86% (GDP obs: 2010-07-01, POT obs: 2011-01-01)
    API HTTP 400 for NROU. Check rate limits.

Fetching vintage data as of 2011-03-15...
  FEDFUNDS=0.16% (obs: 2011-02-01)
  Output gap=-5.70% (GDP obs: 2010-10-01, POT obs: 2011-01-01)
  Unemployment gap=3.70% (UNRATE obs: 2011-02-01, NROU obs: 2011-01-01)

Fetching vintage data as of 2011-04-27...
  FEDFUNDS=0.14% (obs: 2011-03-01)
  Output gap=-6.08% (GDP obs: 2010-10-01, POT obs: 2011-04-01)
  Unemployment gap=3.60% (UNRATE obs: 2011-03-01, NROU obs: 2011-04-01)

Fetching vintage data as of 2011-06-22...
  FEDFUNDS=0.09% (obs: 2011-05-01)
  Output gap=-5.65% (GDP obs: 2011-01-01, POT obs: 2011-04-01)
  Unemployment gap=3.90% (UNRATE obs: 2011-05-01, NROU obs: 2011-04-01)

Fetching vintage data as of 2011-08-09...
  FEDFUNDS=0.07% (obs: 2

In [2]:
import os
import requests
from bs4 import BeautifulSoup
import time
import re

# 1. Configuration
BASE_URL = "https://www.federalreserve.gov"
SAVE_DIR = "/content/drive/MyDrive/HEC Thesis/Text Data/meeting_minutes"
os.makedirs(SAVE_DIR, exist_ok=True)

# Headers to avoid being blocked
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def download_html(url, filename):
    """Downloads the HTML content of a URL and saves it to a file."""
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            return True
    except Exception as e:
        print(f"Error downloading {url}: {e}")
    return False

# --- PART 1: Historical Data (2011-2020) ---
print("Scraping 2011-2020...")
for year in range(2011, 2021):
    hist_url = f"{BASE_URL}/monetarypolicy/fomchistorical{year}.htm"
    print(f"Processing Year: {year}")

    resp = requests.get(hist_url, headers=headers)
    soup = BeautifulSoup(resp.text, 'html.parser')

    # In historical pages, look for 'Minutes (HTML)' links
    links = soup.find_all('a', string=re.compile(r'HTML', re.IGNORECASE))

    for link in links:
        # We only want the ones specifically for 'Minutes'
        parent_text = link.find_parent().get_text() if link.find_parent() else ""
        if 'Minutes' in parent_text:
            href = link.get('href')
            # Extract date from URL (usually fomcminutesYYYYMMDD.htm)
            date_match = re.search(r'(\d{8})', href)
            if date_match:
                date_str = date_match.group(1)
                full_url = f"{BASE_URL}{href}" if href.startswith('/') else f"{BASE_URL}/monetarypolicy/{href}"
                filename = os.path.join(SAVE_DIR, f"minutes_{date_str}.html")

                if not os.path.exists(filename):
                    print(f"  Downloading minutes for {date_str}...")
                    download_html(full_url, filename)
                    time.sleep(1) # Be polite to the server

# --- PART 2: Calendar Data (2021-2025) ---
print("\nScraping 2021-2025...")
calendar_url = f"{BASE_URL}/monetarypolicy/fomccalendars.htm"
resp = requests.get(calendar_url, headers=headers)
soup = BeautifulSoup(resp.text, 'html.parser')

# In the current calendar, minutes are inside specific panel-heading/body structures
# We look for links with 'HTML' where the text or parent contains 'Minutes'
links = soup.find_all('a', string=re.compile(r'HTML', re.IGNORECASE))

for link in links:
    href = link.get('href')
    if href and 'fomcminutes' in href:
        date_match = re.search(r'(\d{8})', href)
        if date_match:
            date_str = date_match.group(1)
            # Filter for 2021-2025
            if int(date_str[:4]) >= 2021:
                full_url = f"{BASE_URL}{href}"
                filename = os.path.join(SAVE_DIR, f"minutes_{date_str}.html")

                if not os.path.exists(filename):
                    print(f"  Downloading minutes for {date_str}...")
                    download_html(full_url, filename)
                    time.sleep(1)

print("\nScraping complete.")

Scraping 2011-2020...
Processing Year: 2011
Processing Year: 2012
Processing Year: 2013
Processing Year: 2014
Processing Year: 2015
Processing Year: 2016
Processing Year: 2017
Processing Year: 2018
Processing Year: 2019
Processing Year: 2020

Scraping 2021-2025...

Scraping complete.


In [4]:
import os
import json
from bs4 import BeautifulSoup
import re

# 1. Configuration
INPUT_DIR = "/content/drive/MyDrive/HEC Thesis/Text Data/meeting_minutes"
OUTPUT_DIR = "/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_minutes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def parse_minutes_html(file_path):
    """Parses FOMC minutes HTML with encoding fallback."""

    # Try UTF-8 first, fallback to latin-1 if it fails
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()
    except UnicodeDecodeError:
        with open(file_path, 'r', encoding='latin-1') as f:
            html_content = f.read()

    soup = BeautifulSoup(html_content, 'html.parser')

    # Identify the main content area
    article = soup.find('div', id='article') or soup.find('div', id='content')
    if not article:
        article = soup.find('body')

    paragraphs = article.find_all('p')

    cleaned_paragraphs = []
    for p in paragraphs:
        text = p.get_text(strip=True)
        # Standardize characters to avoid future JSON issues
        if len(text) > 40:
            text = (text.replace('\xa0', ' ')
                        .replace('\x96', '-')  # Manual fix for that en-dash
                        .replace('“', '"')
                        .replace('”', '"')
                        .replace('’', "'"))
            cleaned_paragraphs.append(text)

    # Extract date from filename
    filename = os.path.basename(file_path)
    date_match = re.search(r'(\d{8})', filename)
    date_str = date_match.group(1) if date_match else "unknown"

    return {
        "date": date_str,
        "source_file": filename,
        "paragraph_count": len(cleaned_paragraphs),
        "content": cleaned_paragraphs
    }
# 2. Execution Loop
print(f"Starting conversion of HTML to JSON...")
files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.html')]

for file_name in files:
    input_path = os.path.join(INPUT_DIR, file_name)
    output_file_name = file_name.replace('.html', '.json')
    output_path = os.path.join(OUTPUT_DIR, output_file_name)

    try:
        structured_data = parse_minutes_html(input_path)

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(structured_data, f, indent=4)

        print(f"  ✓ Processed: {file_name}")
    except Exception as e:
        print(f"  ❌ Error processing {file_name}: {e}")

print(f"\nSuccess! Structured JSON files are saved in: {OUTPUT_DIR}")

Starting conversion of HTML to JSON...
  ✓ Processed: minutes_20110126.html
  ✓ Processed: minutes_20110315.html
  ✓ Processed: minutes_20110427.html
  ✓ Processed: minutes_20110622.html
  ✓ Processed: minutes_20110809.html
  ✓ Processed: minutes_20110921.html
  ✓ Processed: minutes_20111102.html
  ✓ Processed: minutes_20111213.html
  ✓ Processed: minutes_20120125.html
  ✓ Processed: minutes_20120313.html
  ✓ Processed: minutes_20120425.html
  ✓ Processed: minutes_20120620.html
  ✓ Processed: minutes_20120801.html
  ✓ Processed: minutes_20120913.html
  ✓ Processed: minutes_20121024.html
  ✓ Processed: minutes_20121212.html
  ✓ Processed: minutes_20130130.html
  ✓ Processed: minutes_20130320.html
  ✓ Processed: minutes_20130501.html
  ✓ Processed: minutes_20130619.html
  ✓ Processed: minutes_20130731.html
  ✓ Processed: minutes_20130918.html
  ✓ Processed: minutes_20131030.html
  ✓ Processed: minutes_20131218.html
  ✓ Processed: minutes_20140129.html
  ✓ Processed: minutes_20140319.html